**Create meeting minutes from an Audio file**

In [5]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [18]:
pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.8 MB/s eta 0:00:00


In [28]:
# imports

import os
import requests
from IPython.display import Markdown, display, update_display
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig, pipeline
import gradio as gr
import torch
from google.colab import userdata

In [29]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

In [30]:
# model

LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

In [31]:
# Sign in to HuggingFace Hub

hf_token = userdata.get('myHF_TOKEN')
login(hf_token, add_to_git_credential=True)


In [32]:
def load_asr():

    pipe = pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-medium.en",
        dtype=torch.float16,
        device='cuda',
        return_timestamps=True
    )
    return pipe

In [33]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [34]:
def load_llm():
    tokenizer = AutoTokenizer.from_pretrained(LLAMA)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        LLAMA,
        device_map="auto",
        quantization_config=quant_config
    )

    return tokenizer, model

In [13]:
asr_pipe = load_asr()
tokenizer, llm_model = load_llm()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cuda


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [40]:
def transcribe(audio):
    result = asr_pipe(audio)
    return result["text"]

In [41]:
def generate_minutes(transcript):

    prompt = f"""
Convert this transcript into structured meeting minutes.

Format:
Summary
Key Points
Decisions
Action Items
Next Steps

Transcript:
{transcript}
"""

    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = llm_model.generate(
        inputs.input_ids,
        max_new_tokens=600,
        temperature=0.3
    )

    minutes = tokenizer.decode(outputs[0], skip_special_tokens=True)
    minutes = minutes.replace(prompt, "").strip()

    return minutes

In [44]:
def generate_pdf(transcript, minutes):

    file_path = "meeting_minutes.pdf"

    transcript = transcript.replace("\n", "<br/>")
    minutes = minutes.replace("\n", "<br/>")

    doc = SimpleDocTemplate(file_path)
    styles = getSampleStyleSheet()

    content = []

    content.append(Paragraph("<b>Meeting Transcript</b>", styles["Heading2"]))
    content.append(Spacer(1, 10))
    content.append(Paragraph(transcript, styles["Normal"]))

    content.append(Spacer(1, 20))

    content.append(Paragraph("<b>Meeting Minutes</b>", styles["Heading2"]))
    content.append(Spacer(1, 10))
    content.append(Paragraph(minutes, styles["Normal"]))

    doc.build(content)

    return file_path

In [48]:
with gr.Blocks() as demo:

    gr.Markdown("# Meeting Minutes Generator with AI")
    gr.Markdown("Upload or record audio and generate meeting minutes")
    gr.Markdown("---")
    gr.Markdown("Project Developed by Gayathri VR")

    audio_input = gr.Audio(type="filepath")
    transcribe_btn = gr.Button("1️⃣ Transcribe Audio")

    transcript_box = gr.Textbox(label="Transcript", lines=10)

    summarize_btn = gr.Button("2️⃣ Generate Minutes")
    minutes_box = gr.Textbox(label="Meeting Minutes", lines=15)

    pdf_btn = gr.Button("3️⃣ Download PDF")
    pdf_output = gr.File(label="📥 Download PDF")

    # Flow connections
    transcribe_btn.click(transcribe, inputs=audio_input, outputs=transcript_box)

    summarize_btn.click(generate_minutes, inputs=transcript_box, outputs=minutes_box)

    pdf_btn.click(generate_pdf, inputs=[transcript_box, minutes_box], outputs=pdf_output)

In [49]:
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0acbb155d24b7f929d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [36]:
def process_audio(audio_path):

    result = asr_pipe(audio_path)
    transcript = result["text"]
    chunks = result.get("chunks", [])

    formatted_transcript = "\n".join([
        f"{c['timestamp']} : {c['text']}" for c in chunks
    ])

    prompt = f"""
You are a professional AI meeting assistant.

Convert the transcript into formal meeting minutes using EXACT structure below.

Return output in this format ONLY:

---

**Meeting Title:** (if not available, write "General Meeting")
**Date:** (extract if possible, else "Not specified")
**Location:** (if available, else "Not specified")

**Attendees:**
- List names if mentioned, else write "Not specified"

---

**Summary:**
Write a clear 5–6 line professional summary.

---

**Key Discussion Points:**
- Point 1
- Point 2
- Point 3

---

**Decisions Made:**
- Decision 1
- Decision 2

---

**Action Items:**
- Person | Task | Deadline (if not available, write "Not specified")

---

**Announcements:**
- Include events, invitations, or notices

---

**Next Steps:**
- Step 1
- Step 2

---

Transcript:
{formatted_transcript}
"""

    device = "cuda" if torch.cuda.is_available() else "cpu"

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = llm_model.generate(
        inputs.input_ids,
        max_new_tokens=800,
        temperature=0.7,
        top_p=0.9
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return transcript, response

In [38]:
with gr.Blocks() as demo:

    gr.Markdown("# Meeting Minutes Generator with AI")
    gr.Markdown("Upload or record audio and generate meeting minutes")
    gr.Markdown("---")
    gr.Markdown("Project Developed by Gayathri VR")

    audio_input = gr.Audio(type="filepath", label="Upload / Record Audio")

    generate_btn = gr.Button("Generate Minutes")

    transcript_output = gr.Textbox(label="Transcript", lines=10)
    minutes_output = gr.Textbox(label="Meeting Minutes", lines=20)

    pdf_output = gr.File(label="📥 Download PDF")

    generate_btn.click(
        fn=process_audio,
        inputs=audio_input,
        outputs=[transcript_output, minutes_output, pdf_output]
    )

In [20]:
system_message = """
You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.
"""

user_prompt = f"""
Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
{transcription}
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]


In [21]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [22]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 03 Apr 2026

You produce minutes of meetings from transcripts, with summary, key discussion points,
takeaways and action items with owners, in markdown format without code blocks.<|eot_id|><|start_header_id|>user<|end_header_id|>

Below is an extract transcript of a Denver council meeting.
Please write minutes in markdown without code blocks, including:
- a summary with attendees, location and date
- discussion points
- takeaways
- action items with owners

Transcription:
 kind of the confluence of this whole idea of a Confluence Week, the merging of two rivers, and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now, and it's a very big issue. So that is the reason that the back of the logo is considered water. So I'll let you see the creation of the logo here. Yeah, so that basically kind of sums up

In [24]:
response = tokenizer.decode(outputs[0])